In [1]:
!pip install -q praat-parselmouth speechbrain soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 912.7/912.7 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.1/782.1 kB 18.5 MB/s eta 0:00:00


In [2]:
import os, glob, subprocess, numpy as np, torch, soundfile as sf
from collections import Counter
from speechbrain.inference.enhancement import SpectralMaskEnhancement  # older versions: speechbrain.pretrained
torch.set_num_threads(os.cpu_count())

IEM = "/kaggle/input/datasets/jiten597/iemocap/IEMOCAP_full_release"
MELD = "/kaggle/input/datasets/zaber666/meld-dataset/MELD-RAW"
OUT = "/kaggle/working/denoised"

iem = sorted(f for f in glob.glob(f"{IEM}/Session*/sentences/wav/*/*.wav")
             if not os.path.basename(f).startswith("._"))
meld = sorted(f for f in glob.glob(f"{MELD}/**/*.mp4", recursive=True)
              if not os.path.basename(f).startswith("._"))
print("IEMOCAP utterances:", len(iem))
print("MELD mp4:", len(meld))
print(Counter(os.path.relpath(os.path.dirname(f), MELD) for f in meld))

enh = SpectralMaskEnhancement.from_hparams(
    source="speechbrain/metricgan-plus-voicebank", savedir="/kaggle/working/mgan", run_opts={"device": "cpu"})

IEMOCAP utterances: 10039
MELD mp4: 13848
Counter({'MELD.Raw/train/train_splits': 9989, 'MELD.Raw/test/output_repeated_splits_test': 2747, 'MELD.Raw/dev/dev_splits_complete': 1112})


hyperparams.yaml:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

enhance_model.ckpt:   0%|          | 0.00/7.59M [00:00<?, ?B/s]

In [3]:
import time, traceback

def load16k(path):
    if path.endswith(".mp4"):
        raw = subprocess.run(["ffmpeg","-v","error","-i",path,"-ac","1","-ar","16000","-f","s16le","-"],
                             capture_output=True).stdout
        return np.frombuffer(raw, np.int16).astype(np.float32) / 32768
    x, sr = sf.read(path, dtype="float32")
    assert sr == 16000
    return x if x.ndim == 1 else x.mean(1)

def denoise_one(path, out_path):
    x = load16k(path)
    if len(x) < 1600: return False
    with torch.no_grad():
        y = enh.enhance_batch(torch.from_numpy(x)[None], lengths=torch.tensor([1.0]))[0].numpy()
    y *= (np.sqrt(np.mean(x**2)) + 1e-9) / (np.sqrt(np.mean(y**2)) + 1e-9)  # level-match to raw RMS
    y = np.clip(y, -1, 1)
    sf.write(out_path, y, 16000, subtype="PCM_16")
    return True

def run(files, name, key, every=100):
    os.makedirs(f"{OUT}/{name}", exist_ok=True)
    total = len(files)
    t0, fails, done_now = time.time(), [], 0
    already = sum(os.path.exists(f"{OUT}/{name}/{key(f)}.wav") for f in files)
    print(f"[{name}] total={total} already_done={already} to_do={total-already}", flush=True)
    for f in files:
        o = f"{OUT}/{name}/{key(f)}.wav"
        if os.path.exists(o): continue
        try: denoise_one(f, o)
        except Exception as e: fails.append((f, repr(e)))
        done_now += 1
        if done_now % every == 0:
            completed = already + done_now
            left = total - completed
            rate = done_now / (time.time() - t0)
            print(f"[{name}] completed {completed}/{total} ({100*completed/total:.1f}%) | "
                  f"left {left} | {rate:.1f} clips/s | ETA {left/rate/60:.0f} min | fails {len(fails)}",
                  flush=True)
    open(f"{OUT}/{name}_fails.txt","w").write("\n".join(map(str, fails)))
    print(f"[{name}] DONE. completed {total-len(fails)}/{total}, fails {len(fails)}", flush=True)

run(iem, "iemocap", lambda f: os.path.splitext(os.path.basename(f))[0])
run(meld, "meld", lambda f: os.path.relpath(f, MELD).replace("/", "__").replace(".mp4", ""))

[iemocap] total=10039 already_done=0 to_do=10039
[iemocap] completed 100/10039 (1.0%) | left 9939 | 2.8 clips/s | ETA 59 min | fails 0
[iemocap] completed 200/10039 (2.0%) | left 9839 | 3.1 clips/s | ETA 54 min | fails 0
[iemocap] completed 300/10039 (3.0%) | left 9739 | 3.0 clips/s | ETA 54 min | fails 0
[iemocap] completed 400/10039 (4.0%) | left 9639 | 2.9 clips/s | ETA 55 min | fails 0
[iemocap] completed 500/10039 (5.0%) | left 9539 | 2.8 clips/s | ETA 56 min | fails 0
[iemocap] completed 600/10039 (6.0%) | left 9439 | 2.8 clips/s | ETA 56 min | fails 0
[iemocap] completed 700/10039 (7.0%) | left 9339 | 2.8 clips/s | ETA 56 min | fails 0
[iemocap] completed 800/10039 (8.0%) | left 9239 | 2.8 clips/s | ETA 55 min | fails 0
[iemocap] completed 900/10039 (9.0%) | left 9139 | 2.8 clips/s | ETA 54 min | fails 0
[iemocap] completed 1000/10039 (10.0%) | left 9039 | 2.9 clips/s | ETA 52 min | fails 0
[iemocap] completed 1100/10039 (11.0%) | left 8939 | 2.9 clips/s | ETA 51 min | fails 0
[